This notebook is used to finetune the llm to classify the subject mail to Phishing or Legitimate.
Inorder to finetune I have used Qwen model series.


Qwen is a strong choice for fine-tuning phishing vs legitimate classification because it consistently outperforms many open-source peers in real-world adaptability, multilingual understanding, and structured reasoning. Compared to Llama, Mistral, and other mid-size LLMs, Qwen shows better balance between classification accuracy and semantic understanding, especially on noisy security-related text like emails, URLs, and social engineering content.

Key comparative points:
vs Llama (e.g., Llama 3 series):
Llama models are more stable and have a stronger ecosystem, but they often require more fine-tuning data to reach domain-specific performance. Qwen generally shows stronger multilingual handling and slightly better benchmark performance in reasoning-heavy classification tasks, making it more efficient for phishing detection where language variation matters.

vs Mistral:
Mistral is highly efficient and fast, especially in smaller sizes, but it can be less robust in complex domain adaptation. Studies show Mistral may struggle more under aggressive fine-tuning or domain shifts, while Qwen tends to generalize better on diverse real-world inputs and maintains stronger consistency in classification tasks.

vs smaller instruction-tuned models (Phi, Gemma, etc.):
These are lightweight and cheap but lack deep contextual reasoning. Qwen provides a clear performance jump in understanding intent, urgency cues, and semantic phishing patterns, which are critical for security classification.

Why Qwen is especially good for phishing classification:
Learns intent-based signals (urgency, impersonation, manipulation)
Handles multilingual and noisy data better than most peers
Maintains stable decision boundaries after fine-tuning
Supports dual-use setup (classification + explanation generation in one model)



In [1]:
!pip install -q transformers peft bitsandbytes accelerate trl pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer
import warnings
warnings.filterwarnings("ignore")

print("Libraries Ready!")

Libraries Ready!


In [3]:
# Load the dataset
train_df = pd.read_csv("data/train_df.csv", usecols=['subject', 'label','text'])

FileNotFoundError: [Errno 2] No such file or directory: 'data/train_df.csv'

In [ ]:
train_df.head(1)

I chose Qwen2.5-3B-Instruct because it provides the best balance between reasoning ability, fine-tuning efficiency, and deployment feasibility compared to other models like LLaMA or Mistral in the same size range.

Unlike smaller models, it is strong enough to capture semantic patterns in phishing detection (intent, urgency, impersonation signals), and unlike larger Qwen variants (7B+), it is lightweight enough to fine-tune efficiently on limited GPU resources using QLoRA. It is also instruction-tuned, which makes it naturally better at structured classification tasks and consistent outputs, reducing the need for heavy prompt engineering.

Overall, it was selected because it offers a practical production trade-off: high-quality classification performance with low compute cost and fast training cycles.

In [ ]:
# Initializing the model name and setting up the model configurations
model_name = "Qwen/Qwen2.5-3B-Instruct"


In [ ]:
# 4-bit quantization configuration to reduce memory usage and enable training on limited GPU resources
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# Load tokenizer aligned with the Qwen model vocabulary and preprocessing rules
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)


In [ ]:
# Load pretrained Qwen model with 4-bit quantization for efficient fine-tuning
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

In [ ]:
# Prepares model layers for low-bit training (freezes unstable layers and optimizes gradients)
model = prepare_model_for_kbit_training(model)

In [ ]:
# LoRA configuration: trains only small adapter layers instead of full model

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)


In [ ]:
# Wrap model with LoRA adapters (only adapter weights will be trained)
model = get_peft_model(model, lora_config)

In [ ]:
# Initiating the Pipeline for Finetuning

# Setting up the learning configurations
training_args = TrainingArguments(
    output_dir="./qwen_phishing_classifier",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=2,
    fp16=False,
    bf16=True,
    save_strategy="epoch",
    logging_steps=10,
    optim="paged_adamw_8bit",
    report_to="none",
)

In [ ]:
# Converting the pandas dataframe into compatible Trainable Vector
dataset = Dataset.from_pandas(train_df[['text']])

# Initializing the Training Pipeline

# Bypass function to pass for the trainer, due to libary issue
def formatting_func(example):
    return example["text"]
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    formatting_func=formatting_func,
    args=training_args,
)

In [ ]:
print("Starting fine-tuning of classifier...")
trainer.train()
print("Fine tuning complete")

In [ ]:
# change the file path as you needed to save your model and
trainer.save_model("./qwen_phishing_classifier")
tokenizer.save_pretrained("./qwen_phishing_classifier")